# sage実行とmzMLファイル検査

**対応記事**: [article-04b-sage-execution.md](../blog/article-04b-sage-execution.md)  
**実行順序**: 4b番目  
**所要時間**: 約60分（sage実行時間含む）

---

## このNotebookで行うこと

sage-proteomicsを実行してDIA-MSデータを解析し、ペプチドレベルの定量結果（lfq.tsv）を生成します。

- mzMLファイルの整合性確認（pymzml使用）
- sage-proteomics実行とログ解析
- 出力ファイル（lfq.tsv）の構造確認
- 処理性能の評価

## 前提条件

- [notebook_04a_sage_fundamentals.ipynb](./notebook_04a_sage_fundamentals.ipynb) が完了していること
- sage設定ファイル（`sage_config.json`）が作成済み
- ヒトプロテオームFASTAファイルが準備済み
- mzMLファイルが `data/raw/raw_mzML/` に配置済み

## 1. 環境準備とライブラリ導入

In [ ]:
# 標準ライブラリ: ファイル操作(os)、パターン検索(glob)、正規表現(re)、時間計測(time)、外部コマンド実行(subprocess)、システム操作(sys)
import os, glob, re, time, subprocess, sys
import pandas as pd   # データフレーム操作ライブラリ（表形式データの読み書き・加工に使う）
import numpy as np    # 数値計算ライブラリ（配列操作・数学関数に使用）
from pathlib import Path  # モダンなパス操作用ライブラリ
import shutil         # ファイル操作・コマンド存在確認用

# プロテオミクス特化ライブラリ
try:
    import pymzml     # mzMLファイル（質量分析データ）をPythonで読み込むためのライブラリ
    PYMZML_AVAILABLE = True
    print("✅ pymzml利用可能")
except ImportError:
    PYMZML_AVAILABLE = False
    print("❌ pymzml未インストール: mzMLファイル検査をスキップします")
    print("インストール: pip install pymzml")

# sage-proteomicsコマンドの存在確認
SAGE_AVAILABLE = shutil.which("sage") is not None
if SAGE_AVAILABLE:
    print("✅ sage-proteomics利用可能")
else:
    print("❌ sage-proteomics未インストール")
    print("インストール: pip install sage-proteomics")

In [ ]:
# --- パス設定（プロジェクト内の各ディレクトリ・ファイルへの相対パスを定数として定義） ---

# 基本ディレクトリ設定
PROJECT_ROOT = Path("..").resolve()  # notebooksディレクトリから1つ上のレベル
MZML_DIR = PROJECT_ROOT / "data" / "raw" / "raw_mzML"  # mzML生データが格納されているディレクトリ
FASTA_PATH = PROJECT_ROOT / "data" / "raw" / "human_proteome.fasta"  # ヒトプロテオームのFASTA配列ファイルのパス
RESULTS_DIR = PROJECT_ROOT / "results"  # 解析結果の出力先ルートディレクトリ
SAGE_CONFIG = PROJECT_ROOT / "scripts" / "sage_config.json"  # sageの設定ファイル（検索パラメータを記述）

# 出力先ディレクトリ
SAGE_OUT = RESULTS_DIR / "sage_output"  # sageの出力先ディレクトリ（lfq.tsvなどが生成される）
TABLES_DIR = RESULTS_DIR / "tables"  # 検査結果テーブルの保存先ディレクトリ
FIGURES_DIR = RESULTS_DIR / "figures"  # 図表の保存先ディレクトリ

# パス表示
print("=== パス設定確認 ===")
print(f"プロジェクトルート: {PROJECT_ROOT}")
print(f"mzMLディレクトリ: {MZML_DIR}")
print(f"FASTAファイル: {FASTA_PATH}")
print(f"sage設定ファイル: {SAGE_CONFIG}")
print(f"sage出力先: {SAGE_OUT}")

# 出力先ディレクトリが存在しなければ作成する（exist_ok=Trueで既存でもエラーにならない）
for directory in [RESULTS_DIR, SAGE_OUT, TABLES_DIR, FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"✅ ディレクトリ確認/作成: {directory}")

## 2. 入力ファイルの存在確認

In [ ]:
# 必要なファイルの存在確認
print("=== 入力ファイル存在確認 ===")

# mzMLディレクトリとファイル数確認
if MZML_DIR.exists():
    mzml_files = list(MZML_DIR.glob("*.mzML")) + list(MZML_DIR.glob("*.mzml"))
    print(f"✅ mzMLディレクトリ存在: {len(mzml_files)} ファイル")
    if len(mzml_files) == 0:
        print("⚠️ mzMLファイルが見つかりません。データダウンロードを確認してください。")
else:
    print(f"❌ mzMLディレクトリなし: {MZML_DIR}")

# FASTAファイル確認
if FASTA_PATH.exists():
    fasta_size_mb = FASTA_PATH.stat().st_size / 1024**2
    print(f"✅ FASTAファイル存在: {fasta_size_mb:.1f} MB")
else:
    print(f"❌ FASTAファイルなし: {FASTA_PATH}")

# sage設定ファイル確認
if SAGE_CONFIG.exists():
    print(f"✅ sage設定ファイル存在")
    # 設定ファイルの一部を表示
    with open(SAGE_CONFIG, 'r') as f:
        config_preview = f.read()[:200]  # 最初の200文字
    print(f"設定プレビュー: {config_preview}...")
else:
    print(f"❌ sage設定ファイルなし: {SAGE_CONFIG}")

# 実行可能性の判定
can_run_analysis = (
    MZML_DIR.exists() and len(mzml_files) > 0 and
    FASTA_PATH.exists() and
    SAGE_CONFIG.exists() and
    SAGE_AVAILABLE
)

print(f"\n=== 実行可能性判定 ===")
if can_run_analysis:
    print("🎉 sage解析実行可能")
else:
    print("❌ sage解析実行不可能 - 上記の問題を解決してください")

## 3. mzMLファイルの検査

sage実行前に、入力mzMLファイルの整合性を確認します。各ファイルのMS1/MS2スペクトル数や保持時間（RT）範囲をチェックします。

In [ ]:
def inspect_mzml(path):
    """mzMLを走査してMS1/MS2数とRT範囲を返す。
    
    Args:
        path (str): mzMLファイルのパス
        
    Returns:
        dict: ファイル情報（ファイル名、サイズ、スペクトル数、RT範囲）
    """
    try:
        # pymzmlでmzMLファイルを開き、スペクトルを1枚ずつ読み込めるReaderオブジェクトを作成
        reader = pymzml.run.Reader(str(path))
        ms1, ms2 = 0, 0  # MS1（前駆体スキャン）とMS2（フラグメントスキャン）のカウンターを初期化
        # 保持時間（RT）の最小値・最大値を追跡するための初期値（infは無限大）
        rt_min, rt_max = float("inf"), float("-inf")

        # mzMLファイル内の全スペクトルを1枚ずつ走査するループ
        for spec in reader:
            rt = spec.scan_time_in_minutes()  # そのスペクトルの保持時間（分単位）を取得
            if rt is not None:
                # 保持時間が取得できた場合、最小値・最大値を更新
                rt_min, rt_max = min(rt_min, rt), max(rt_max, rt)
            if spec.ms_level == 1:    # MS1スペクトル（前駆体イオンの全体像）の場合
                ms1 += 1              # MS1カウンターを加算
            elif spec.ms_level == 2:  # MS2スペクトル（フラグメントイオン）の場合
                ms2 += 1              # MS2カウンターを加算

        # 検査結果を辞書にまとめて返す（後でDataFrameの1行になる）
        return {
            "file": path.name,  # ファイル名のみ（ディレクトリパスを除去）
            "size_MB": round(path.stat().st_size / 1024**2, 1),  # ファイルサイズをMB単位に変換（小数1桁）
            "ms1": ms1,                    # MS1スペクトルの総数
            "ms2": ms2,                    # MS2スペクトルの総数
            "rt_min": round(rt_min, 2) if rt_min != float("inf") else 0,    # 保持時間の最小値（分、小数2桁）
            "rt_max": round(rt_max, 2) if rt_max != float("-inf") else 0,   # 保持時間の最大値（分、小数2桁）
            "status": "OK"
        }
    except Exception as e:
        # mzMLファイルに問題がある場合のエラーハンドリング
        return {
            "file": path.name,
            "size_MB": round(path.stat().st_size / 1024**2, 1) if path.exists() else 0,
            "ms1": 0,
            "ms2": 0,
            "rt_min": 0,
            "rt_max": 0,
            "status": f"Error: {str(e)[:50]}"  # エラーメッセージ（最初の50文字）
        }

print("✅ mzMLファイル検査関数定義完了")

In [ ]:
# mzMLファイル検査の実行
if PYMZML_AVAILABLE and MZML_DIR.exists():
    print("=== mzMLファイル検査開始 ===")
    
    # mzMLファイル一覧を取得して検査を実行
    # MZML_DIRディレクトリ内の全.mzMLファイルをglobで検索し、ファイル名順にソート
    mzml_files = sorted(MZML_DIR.glob("*.mzML")) + sorted(MZML_DIR.glob("*.mzml"))
    print(f"{len(mzml_files)} mzMLファイルを検査中...")
    
    # 各ファイルを検査（時間がかかる処理なので進捗表示）
    records = []
    for i, f in enumerate(mzml_files):
        print(f"処理中 ({i+1}/{len(mzml_files)}): {f.name}")
        record = inspect_mzml(f)
        records.append(record)
    
    # 検査結果のリスト（辞書のリスト）をpandas DataFrameに変換（表形式にする）
    df_mzml_inventory = pd.DataFrame(records)
    
    # 検査結果をCSVファイルとして保存（index=Falseで行番号は含めない）
    inventory_path = TABLES_DIR / "mzml_inventory.csv"
    df_mzml_inventory.to_csv(inventory_path, index=False)
    print(f"✅ 検査結果保存: {inventory_path}")
    
    # 検査結果の表示
    print("\n=== mzMLファイル検査結果 ===")
    display(df_mzml_inventory)
    
    # 基本統計
    print("\n=== 検査統計 ===")
    total_size_gb = df_mzml_inventory['size_MB'].sum() / 1024
    total_ms1 = df_mzml_inventory['ms1'].sum()
    total_ms2 = df_mzml_inventory['ms2'].sum()
    ok_files = len(df_mzml_inventory[df_mzml_inventory['status'] == 'OK'])
    
    print(f"総ファイル数: {len(df_mzml_inventory)}")
    print(f"正常ファイル数: {ok_files}")
    print(f"総サイズ: {total_size_gb:.2f} GB")
    print(f"総MS1スペクトル: {total_ms1:,}")
    print(f"総MS2スペクトル: {total_ms2:,}")
    
    # エラーファイルがあれば警告
    error_files = df_mzml_inventory[df_mzml_inventory['status'] != 'OK']
    if len(error_files) > 0:
        print(f"\n⚠️ 問題があるファイル ({len(error_files)} 個):")
        for _, row in error_files.iterrows():
            print(f"  - {row['file']}: {row['status']}")
    else:
        print("\n✅ 全ファイル正常")
        
else:
    print("⚠️ pymzml未使用またはmzMLファイル不足のため、検査をスキップ")
    df_mzml_inventory = None

## 4. sage-proteomics実行

sage をコマンドラインではなく Python の `subprocess` 経由で実行します。実行時間の計測やログのリアルタイム表示を Notebook 内で完結できます。

In [ ]:
# sage実行の準備
if can_run_analysis:
    print("=== sage実行準備 ===")
    
    # sageコマンドの引数をリストとして構築（subprocessに渡す形式）
    mzml_pattern = str(MZML_DIR / "*.mzML")
    mzml_files_list = list(MZML_DIR.glob("*.mzML")) + list(MZML_DIR.glob("*.mzml"))
    
    sage_cmd = [
        "sage",
        "--fasta", str(FASTA_PATH),  # FASTAファイルパス
        "--output_directory", str(SAGE_OUT),  # 出力ディレクトリ
        "--disable-telemetry-i-dont-want-to-improve-sage",  # テレメトリ無効化
        str(SAGE_CONFIG)  # 設定ファイル
    ] + [str(f) for f in mzml_files_list]  # 全mzMLファイルを追加
    
    print(f"sage実行予定コマンド:")
    print(f"sage \\")
    print(f"  --fasta {FASTA_PATH} \\")
    print(f"  --output_directory {SAGE_OUT} \\")
    print(f"  --disable-telemetry-i-dont-want-to-improve-sage \\")
    print(f"  {SAGE_CONFIG} \\")
    print(f"  {len(mzml_files_list)} mzMLファイル")
    
    print(f"\n処理対象ファイル数: {len(mzml_files_list)}")
    print(f"予想処理時間: 約10-15分")
    
else:
    print("❌ sage実行不可能 - 前提条件を満たしていません")

In [ ]:
# sage実行（長時間処理）
if can_run_analysis:
    print("=== sage実行開始 ===")
    print("⏰ 処理には約10-15分かかります...\n")
    
    # 実行開始時刻を記録
    start_time = time.time()
    
    try:
        # sageをsubprocessで実行（リアルタイム出力表示）
        process = subprocess.Popen(
            sage_cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,  # stderrもstdoutにリダイレクト
            text=True,
            bufsize=1,  # 行バッファリング
            universal_newlines=True
        )
        
        # リアルタイム出力の表示
        output_lines = []
        while True:
            line = process.stdout.readline()
            if line:
                print(line.rstrip())  # 改行を除去して表示
                output_lines.append(line.rstrip())
            elif process.poll() is not None:  # プロセス終了
                break
        
        # 残りの出力を取得
        remaining_output, _ = process.communicate()
        if remaining_output:
            for line in remaining_output.split('\n'):
                if line.strip():
                    print(line)
                    output_lines.append(line)
        
        # 実行時間計算
        end_time = time.time()
        execution_time_seconds = end_time - start_time
        execution_time_minutes = execution_time_seconds / 60
        
        # 結果判定
        if process.returncode == 0:
            print(f"\n🎉 sage実行完了!")
            print(f"⏱️ 実行時間: {execution_time_minutes:.1f}分 ({execution_time_seconds:.1f}秒)")
            sage_success = True
        else:
            print(f"\n❌ sage実行失敗 (終了コード: {process.returncode})")
            sage_success = False
            
        # ログを保存
        log_path = SAGE_OUT / "sage_execution_log.txt"
        with open(log_path, 'w') as f:
            f.write(f"Sage execution log\n")
            f.write(f"Start time: {time.ctime(start_time)}\n")
            f.write(f"End time: {time.ctime(end_time)}\n")
            f.write(f"Execution time: {execution_time_minutes:.1f} minutes\n")
            f.write(f"Return code: {process.returncode}\n")
            f.write("\n=== OUTPUT ===\n")
            f.write('\n'.join(output_lines))
        
        print(f"📝 実行ログ保存: {log_path}")
        
    except Exception as e:
        print(f"\n❌ sage実行中にエラーが発生: {e}")
        sage_success = False
        
else:
    print("⚠️ sage実行をスキップ（前提条件未満）")
    sage_success = False

## 5. 出力ファイルの確認と解析

In [ ]:
# sage出力ファイルの確認
print("=== sage出力ファイル確認 ===")

# 期待される出力ファイル
expected_outputs = {
    'lfq.tsv': 'ペプチド × サンプル定量マトリクス',
    'results.sage.tsv': 'PSM（ペプチドスペクトラムマッチ）テーブル',
    'results.json': 'パラメータ記録とメタデータ'
}

output_files_status = {}

for filename, description in expected_outputs.items():
    filepath = SAGE_OUT / filename
    if filepath.exists():
        size_mb = filepath.stat().st_size / 1024**2
        print(f"✅ {filename}: {size_mb:.1f} MB ({description})")
        output_files_status[filename] = {'exists': True, 'size_mb': size_mb, 'path': filepath}
    else:
        print(f"❌ {filename}: 見つかりません ({description})")
        output_files_status[filename] = {'exists': False, 'size_mb': 0, 'path': filepath}

# lfq.tsvが存在する場合、詳細解析
lfq_path = SAGE_OUT / "lfq.tsv"
if lfq_path.exists():
    print(f"\n=== lfq.tsv詳細解析 ===")
    
    # ファイルの先頭部分を読み込み（全体は大きすぎる可能性があるため）
    df_lfq_preview = pd.read_csv(lfq_path, sep='\t', nrows=10)  # 最初の10行のみ
    
    print(f"カラム数: {len(df_lfq_preview.columns)}")
    print(f"主要カラム: {', '.join(df_lfq_preview.columns[:8].tolist())}")
    
    # 全行数を取得（メモリ効率的に）
    with open(lfq_path, 'r') as f:
        line_count = sum(1 for line in f) - 1  # ヘッダー行を除く
    print(f"総ペプチド数: {line_count:,}")
    
    # プレビュー表示
    print("\n=== lfq.tsvプレビュー（先頭5行） ===")
    display(df_lfq_preview.head())
    
    # サンプル列の抽出（.mzMLで終わるカラム）
    sample_columns = [col for col in df_lfq_preview.columns if col.endswith('.mzML')]
    print(f"\n検出サンプル数: {len(sample_columns)}")
    if len(sample_columns) > 0:
        print(f"サンプル例: {', '.join(sample_columns[:5])}")
        if len(sample_columns) > 5:
            print(f"... 他 {len(sample_columns) - 5} サンプル")
            
else:
    print("⚠️ lfq.tsvが見つかりません - sage実行が失敗した可能性があります")


## 6. 処理性能の分析

In [ ]:
# sage実行ログから性能指標を抽出
log_path = SAGE_OUT / "sage_execution_log.txt"

if log_path.exists():
    print("=== 処理性能分析 ===")
    
    with open(log_path, 'r') as f:
        log_content = f.read()
    
    # ログから重要な指標を抽出
    performance_metrics = {}
    
    # 実行時間の抽出
    if "Execution time:" in log_content:
        for line in log_content.split('\n'):
            if "Execution time:" in line:
                time_match = re.search(r'(\d+\.\d+) minutes', line)
                if time_match:
                    performance_metrics['execution_time_min'] = float(time_match.group(1))
    
    # フラグメント数とペプチド数の抽出
    fragment_match = re.search(r'generated (\d+) fragments, (\d+) peptides in (\d+)ms', log_content)
    if fragment_match:
        performance_metrics['fragments'] = int(fragment_match.group(1))
        performance_metrics['peptides'] = int(fragment_match.group(2))
        performance_metrics['generation_time_ms'] = int(fragment_match.group(3))
    
    # 検索速度の抽出
    search_match = re.search(r'search:\s+(\d+) ms \((\d+) spectra/s\)', log_content)
    if search_match:
        performance_metrics['search_time_ms'] = int(search_match.group(1))
        performance_metrics['spectra_per_second'] = int(search_match.group(2))
    
    # 性能サマリー表示
    print("📊 **処理性能サマリー**")
    
    if 'execution_time_min' in performance_metrics:
        exec_time = performance_metrics['execution_time_min']
        print(f"⏱️ 総実行時間: {exec_time:.1f}分")
        
        # ファイル数がわかれば1ファイルあたりの時間を計算
        if 'mzml_files_list' in locals():
            time_per_file = exec_time / len(mzml_files_list)
            print(f"📁 1ファイルあたり: {time_per_file:.2f}分")
    
    if 'fragments' in performance_metrics:
        fragments = performance_metrics['fragments']
        peptides = performance_metrics['peptides']
        gen_time = performance_metrics['generation_time_ms'] / 1000
        print(f"🧬 生成フラグメント: {fragments:,}")
        print(f"🧬 生成ペプチド: {peptides:,}")
        print(f"⚡ 生成時間: {gen_time:.1f}秒")
    
    if 'spectra_per_second' in performance_metrics:
        sps = performance_metrics['spectra_per_second']
        search_time = performance_metrics['search_time_ms'] / 1000
        print(f"🔍 検索速度: {sps:,} spectra/秒")
        print(f"🔍 検索時間: {search_time:.1f}秒")
    
    # 論文との比較
    print(f"\n📋 **論文との性能比較**")
    print(f"論文記載: 約10.7分で32ファイル処理")
    if 'execution_time_min' in performance_metrics:
        print(f"本実行: {performance_metrics['execution_time_min']:.1f}分で{len(mzml_files_list) if 'mzml_files_list' in locals() else '?'}ファイル処理")
        if performance_metrics['execution_time_min'] <= 12:
            print("✅ 論文記載の性能を達成")
        else:
            print("⚠️ 論文記載より時間がかかりました")

else:
    print("⚠️ 実行ログが見つかりません")

## まとめ

✅ **完了項目**:
- mzMLファイルの整合性確認
- sage-proteomics実行によるペプチド同定・定量
- 出力ファイル（lfq.tsv）の生成確認
- 処理性能の評価

**主要な出力**:
- `lfq.tsv`: ペプチドレベル定量マトリクス
- `results.sage.tsv`: PSM詳細結果
- `results.json`: 解析パラメータ記録

**次のステップ**:
- ペプチド→タンパク質レベルの集約
- 遺伝子名マッピング
- プロテインマトリクスの作成

In [ ]:
# 最終ステータス確認
print("=== 最終ステータス ===")

# 重要ファイルの存在確認
critical_outputs = {
    'lfq.tsv': SAGE_OUT / 'lfq.tsv',
    'mzml_inventory.csv': TABLES_DIR / 'mzml_inventory.csv'
}

all_critical_exist = True
for name, path in critical_outputs.items():
    if path.exists():
        size_mb = path.stat().st_size / 1024**2
        print(f"✅ {name}: {size_mb:.1f} MB")
    else:
        print(f"❌ {name}: 見つかりません")
        all_critical_exist = False

if all_critical_exist:
    print("\n🎉 すべての重要ファイルが生成されました")
    print("次のNotebook（遺伝子マッピング・プロテインマトリクス作成）に進めます")
else:
    print("\n⚠️ 一部ファイルが不足しています。sage実行を確認してください")

---

## Navigation

⬅️ **前回**: [notebook_04a_sage_fundamentals.ipynb](./notebook_04a_sage_fundamentals.ipynb) — sage基礎とセットアップ  
➡️ **次回**: [notebook_04c_sage_gene_mapping.ipynb](./notebook_04c_sage_gene_mapping.ipynb) — 遺伝子名マッピング

---

*このNotebookは [article-04b-sage-execution.md](../blog/article-04b-sage-execution.md) に対応しています。*